# Imports & Setup

In [44]:
import pandas as pd
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Load initial Data

In [45]:
# Define relative paths based on the notebook location (inside 'notebooks/' folder)
input_file = Path('../data/raw data/weather-data_2025.csv')

# Load the initial CSV file created in the previous preparation steps
df = pd.read_csv(input_file)

# Display the first few rows to verify
df.head()

,timestamp,temperature_2m,shortwave_radiation,Location
0,2025-01-01 00:00:00,-1.5,0.0,Lat:47.34622_Lon:8.543372
1,2025-01-01 01:00:00,-1.9,0.0,Lat:47.34622_Lon:8.543372
2,2025-01-01 02:00:00,-3.2,0.0,Lat:47.34622_Lon:8.543372
3,2025-01-01 03:00:00,-3.1,0.0,Lat:47.34622_Lon:8.543372
4,2025-01-01 04:00:00,-2.9,0.0,Lat:47.34622_Lon:8.543372


# Adjust Timestamps

In [46]:
# Convert timestamp string to datetime objects
# utc=True erzwingt das Erkennen des Offsets, tz_convert setzt es explizit auf Zürich
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert('Europe/Zurich')

# Check the data types to ensure successful conversion (sollte 'datetime64[ns, Europe/Zurich]' sein)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 61320 entries, 0 to 61319
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype                        
---  ------               --------------  -----                        
 0   timestamp            61320 non-null  datetime64[us, Europe/Zurich]
 1   temperature_2m       61320 non-null  float64                      
 2   shortwave_radiation  61320 non-null  float64                      
 3   Location             61320 non-null  str                          
dtypes: datetime64[us, Europe/Zurich](1), float64(2), str(1)
memory usage: 3.4 MB


# Remove Longitude and Latitude Columns

In [47]:
# Safely drop 'latitude' and 'longitude' if they exist in the current dataframe
columns_to_drop = ['latitude', 'longitude']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

# Display updated columns
df.head()


,timestamp,temperature_2m,shortwave_radiation,Location
0,2025-01-01 01:00:00+01:00,-1.5,0.0,Lat:47.34622_Lon:8.543372
1,2025-01-01 02:00:00+01:00,-1.9,0.0,Lat:47.34622_Lon:8.543372
2,2025-01-01 03:00:00+01:00,-3.2,0.0,Lat:47.34622_Lon:8.543372
3,2025-01-01 04:00:00+01:00,-3.1,0.0,Lat:47.34622_Lon:8.543372
4,2025-01-01 05:00:00+01:00,-2.9,0.0,Lat:47.34622_Lon:8.543372


# Aggregate Weather Data

In [48]:
# Group by timestamp, calculate the mean for specific columns, round to 4 decimals, and reset the index
df_avg = df.groupby('timestamp')[['temperature_2m', 'shortwave_radiation']].mean().round(4).reset_index()

# Display the aggregated results
df_avg.head()


,timestamp,temperature_2m,shortwave_radiation
0,2025-01-01 01:00:00+01:00,-1.4571,0.0
1,2025-01-01 02:00:00+01:00,-1.7857,0.0
2,2025-01-01 03:00:00+01:00,-2.4000,0.0
3,2025-01-01 04:00:00+01:00,-2.4714,0.0
4,2025-01-01 05:00:00+01:00,-2.5714,0.0


# Save the Final Processed Data

In [49]:
# Zur Sicherheit: Falls die Zeitzone durch das Groupby verloren ging, hier nochmal setzen
if df_avg['timestamp'].dt.tz is None:
    df_avg['timestamp'] = df_avg['timestamp'].dt.tz_localize('UTC').dt.tz_convert('Europe/Zurich')

# Wandelt den Zeitstempel exakt in das Format YYYY-MM-DDTHH:MM:SS+01:00 um
df_avg['timestamp'] = df_avg['timestamp'].apply(lambda x: x.isoformat(timespec='seconds'))

# Export the cleaned and aggregated dataframe to a new CSV file in the 'clean data' folder
output_file = Path('../data/processed data/weather-data_2025.csv')

# Ensure the output directory exists
output_file.parent.mkdir(parents=True, exist_ok=True)

df_avg.to_csv(output_file, index=False)

print(f'Data successfully saved to {output_file}')

Data successfully saved to ../data/processed data/weather-data_2025.csv
